In [ ]:
import yaml
import pandas as pd
import numpy as np
from tqdm import tqdm
import os

# Consolidate FuncRVP results

## get file paths

In [ ]:
op_base_dir = '/s/project/geno2pheno/funcrvp/paper_revisions/cv'
geno_dir = 'ukbb_wes_500k_DeepRVAT_final_090924_medianshifted'

In [ ]:
# emb_model_list = ['enformer_d512/funcrvp_better_filteredv3_samplingNone', 'ESM2_PCA_d512/funcrvp_better_filteredv3_samplingNone', 'gene2vec_d200/funcrvp_better_filteredv3_samplingNone']
emb_model_list = ['pops_mat_pca256_omics/funcrvp_cv_filteredv3_0/', 'pops_mat_pca256_omics/funcrvp_cv_filteredv3_1/', 'pops_mat_pca256_omics/funcrvp_cv_filteredv3_2/', 'pops_mat_pca256_omics/funcrvp_cv_filteredv3_3/', 'pops_mat_pca256_omics/funcrvp_cv_filteredv3_4/']

model_dir_list = [os.path.join(op_base_dir, geno_dir, model_dir) for model_dir in emb_model_list]
model_dir_list

In [ ]:
# config_path = '../run_config_local.yaml'
config_path = '../run_config_CV.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

traits=config.get('traits')
# traits

## Save consolidated files

In [ ]:
for model_dir in tqdm(model_dir_list):
    beta_trait_list = []
    pred_trait_list = []
    for trait in tqdm(traits):
        # Load the predictions
        beta_file = os.path.join(model_dir, f"{trait}_betas.pq")
        beta_trait_list.append(pd.read_parquet(beta_file))

        pred_file = os.path.join(model_dir, f"{trait}_phenopred.pq")  
        pred_trait_list.append(pd.read_parquet(pred_file))
    
    # Concatenate the dataframes
    beta_df = pd.concat(beta_trait_list, axis=0)
    pred_df = pd.concat(pred_trait_list, axis=0).reset_index()
    # Save the concatenated dataframes to parquet files
    beta_df.to_parquet(os.path.join(model_dir, "all_traits_betas.pq"))
    pred_df.to_parquet(os.path.join(model_dir, "all_traits_phenopred.pq"))


# Save RVAT results

In [ ]:
op_base_dir = '/s/project/geno2pheno/funcrvp/paper_revisions/predictions'
geno_dir = 'ukbb_wes_500k_DeepRVAT_final_090924_medianshifted'
rvat_dir = os.path.join(op_base_dir, geno_dir, 'rvat')

config_path = '../run_config_local.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

traits=config.get('traits')

In [ ]:

beta_trait_list = []
for trait in tqdm(traits):
    beta_file = os.path.join(rvat_dir, f"{trait}_rvat.pq")
    beta_trait_list.append(pd.read_parquet(beta_file))

# Concatenate and save the dataframes
beta_df = pd.concat(beta_trait_list, axis=0)
beta_df.to_parquet(os.path.join(rvat_dir, "all_traits_rvat.pq"))


In [ ]:
# p_thresh_list = ['0.005', '0.01', '0.05', '0.1'] 0.00005
p_thresh_list = ['0.0001', '0.001', '0.01', '0.05', '0.1', '0.25', '0.5', '1.0']

for p_thresh in tqdm(p_thresh_list):
    pred_trait_list = []
    for trait in tqdm(traits):
        pred_file = os.path.join(rvat_dir, f"{trait}_phenopred_{p_thresh}nom.pq")  
        pred_trait_list.append(pd.read_parquet(pred_file))
    
    pred_df = pd.concat(pred_trait_list, axis=0).reset_index()
    # pred_df.to_parquet(os.path.join(rvat_dir, f"all_traits_phenopred_{p_thresh}.pq"))
    pred_df.to_parquet(os.path.join(rvat_dir, f"all_traits_phenopred_{p_thresh}nom.pq"))


# Debug

In [ ]:
import yaml
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
from plotnine import *
from sklearn.metrics import r2_score

In [ ]:
op_base_dir = '/s/project/geno2pheno/funcrvp/paper_revisions/funcrvp_predictions/'
geno_dir = 'ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/'
# model_dir = 'pops_mat_pca256_omics/funcrvp_better_filteredv3/'
# model_dir = 'codon_emb_d64/funcrvp_rev1_filteredv3/'
# emb_model_list = ['omics_d256/funcrvp_rev1_filteredv3', 'pops_mat_d256/funcrvp_rev1_filteredv3', 'codon_emb_d64/funcrvp_rev1_filteredv3', 'tabula_sapiens2_100_000_cells_d512/funcrvp_rev1_filteredv3', 'tabula_sapiens2_pseudobulk_d512/funcrvp_rev1_filteredv3']
emb_model_list = ['rvat']

df_list = []
for model_dir in tqdm(emb_model_list):
    df_list.append(pd.read_parquet(os.path.join(op_base_dir, geno_dir, model_dir, 'all_traits_phenopred.pq')))

dt_new = pd.concat(df_list)
dt_new

In [ ]:
# dt_new_ldl = dt_new[dt_new['trait'] == 'LDL_direct']
r2_per_trait = pd.DataFrame(dt_new.groupby(['embedding', 'trait']).apply(lambda group: r2_score(group['trait_measurement'], group['best_prediction']))).reset_index()
r2_per_trait.columns = ['embedding', 'trait', 'new_r2']
r2_per_trait

In [ ]:
r2_per_trait = r2_per_trait.pivot(index='trait', columns='embedding', values='new_r2').reset_index()
r2_per_trait

In [ ]:
test_size = '0.25'
version = 'v1NEWsplit'
genotype = 'deepRVAT'
emb = 'omics_pops'
dt_old = pd.read_parquet(f'/s/project/geno2pheno/predictions/clean/paper_funcrvp/{version}_{genotype}_testsplit{test_size}_{emb}_predictions_extended.pq')

# dt_old = pd.read_parquet('/s/project/geno2pheno/funcrvp/paper_revisions/debug/old_lm_filteredv3_deepRVAT_0.25_predictions_NEWsplit.pq')

dt_old

In [ ]:
r2_old = pd.DataFrame(dt_old.groupby('trait').apply(lambda group: r2_score(group['trait_measurement'], group['best_r2_pred']))).reset_index()
r2_old.columns = ['trait', 'old_r2']
# r2_old['embedding'] = 'manu_funcrvp'
r2_old

In [ ]:
r2_per_trait = r2_per_trait.merge(r2_old, on='trait', how='left')
r2_per_trait['diff'] = r2_per_trait['new_r2'] - r2_per_trait['old_r2']
r2_per_trait

In [ ]:
r2_per_trait = r2_per_trait.sort_values('diff', ascending=False)
r2_per_trait['trait'] = pd.Categorical(r2_per_trait['trait'], categories=r2_per_trait['trait'].unique(), ordered=True)

(
    ggplot(r2_per_trait, aes(x='trait', y='diff', label='trait')) +
    geom_point() +
    theme_bw() +
    ylab('r2_new - r2_old') +
    facet_wrap('~embedding') +
    theme(
        axis_text_x=element_text(rotation=90),
        figure_size=(16, 10),
    )
)